# GRU model selection — four public forecasting datasets

Validation-based GRU selection for ETTh1, ETTh2, Weather, and Electricity. 


In [ ]:
from pathlib import Path

def find_project_root(start=None):
    """Locate the repository root from the current working directory."""
    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "Data").is_dir() and (candidate / "Results").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate CFE_for_MTS_Forecasting_Framework. "
        "Run this notebook from somewhere inside the cloned repository."
    )

PROJECT_ROOT = find_project_root()
print("Project root:", PROJECT_ROOT)

PUBLIC_ROOT = PROJECT_ROOT / "Data" / "Other"
OUTPUT_ROOT = PROJECT_ROOT / "Results" / "EDBT_public_GRU_v6"

CSV_OVERRIDES = {
    "ETTh1": PUBLIC_ROOT / "ETT-small" / "ETTh1.csv",
    "ETTh2": PUBLIC_ROOT / "ETT-small" / "ETTh2.csv",
    "Electricity": PUBLIC_ROOT / "electricity" / "electricity.csv",
    "Weather": PUBLIC_ROOT / "weather" / "weather_cleaned.csv",
}
OVERWRITE_EXISTING = False
RUN_DATASETS = ("ETTh1", "ETTh2", "Weather", "Electricity")

print("Public data root:", PUBLIC_ROOT, "| exists:", PUBLIC_ROOT.exists())
print("Output root:", OUTPUT_ROOT)


In [ ]:
# 2. Imports and configuration. These match the original piezometric GRU search.
import hashlib, json, math, random, time, warnings
from dataclasses import asdict, dataclass, replace
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

@dataclass(frozen=True)
class GRUConfig:
    input_len: int = 336
    horizon: int = 24
    train_ratio: float = 0.50
    val_ratio: float = 0.10
    batch_size: int = 128
    gru_hidden: int = 100
    gru_layers: int = 2
    gru_dropout: float = 0.0
    gru_lrs: tuple = (1e-4, 3e-4, 1e-5, 3e-5, 1e-3, 3e-3)
    gru_epochs: int = 80
    patience: int = 10
    min_delta: float = 1e-5
    seed: int = 42
    num_workers: int = 0
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

# Shared architecture/training settings; only two time-window fields vary per dataset.
BASE_CFG = GRUConfig()
WINDOW_SPECS = {
    'ETTh1': (336, 24),
    'ETTh2': (336, 24),
    'Weather': (288, 24),
    'Electricity': (168, 24),
}
DATASET_CFG = {name: replace(BASE_CFG, input_len=WINDOW_SPECS[name][0],
                             horizon=WINDOW_SPECS[name][1]) for name in RUN_DATASETS}
assert BASE_CFG.train_ratio + BASE_CFG.val_ratio < 1

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def lr_tag(lr):
    return f'{lr:.0e}'

# Target names are exact CSV headers; no automatic target substitution is allowed.
TARGETS = {'ETTh1': 'OT', 'ETTh2': 'OT', 'Weather': 'T (degC)', 'Electricity': '319'}
EXPECTED_ROWS = {'ETTh1': 17420, 'ETTh2': 17420, 'Weather': 52704, 'Electricity': 26304}
EXPECTED_FEATURES = {'ETTh1': 7, 'ETTh2': 7, 'Weather': 21, 'Electricity': 321}
print('PyTorch:', torch.__version__, '| Device:', BASE_CFG.device)
print('LR grid:', BASE_CFG.gru_lrs)
print('Input and horizon per dataset:', WINDOW_SPECS)

In [ ]:
# 3. Discover exact files: fail on ambiguity, wrong Weather variant, or missing datasets.
def locate_public_csv(name):
    if name in CSV_OVERRIDES:
        p = Path(CSV_OVERRIDES[name])
        if not p.is_file(): raise FileNotFoundError(f'{name}: override does not exist: {p}')
    else:
        wanted = 'weather_cleaned.csv' if name == 'Weather' else f'{name}.csv'
        matches = sorted(p for p in PUBLIC_ROOT.rglob('*.csv') if p.name.lower() == wanted.lower())
        if not matches:
            raise FileNotFoundError(f'{name}: cannot find {wanted} under {PUBLIC_ROOT}. Set CSV_OVERRIDES.')
        if len(matches) != 1:
            raise ValueError(f'{name}: ambiguous CSVs: {matches}. Set CSV_OVERRIDES[{name!r}] explicitly.')
        p = matches[0]
    if name == 'Weather' and p.name.lower() != 'weather_cleaned.csv':
        raise ValueError('Use weather_cleaned.csv for Weather: the characterization and forecasting CSVs must match.')
    return p.resolve()

CSV_PATHS = {name: locate_public_csv(name) for name in RUN_DATASETS}
for name, p in CSV_PATHS.items():
    header = pd.read_csv(p, nrows=0).columns.tolist()
    if TARGETS[name] not in header:
        raise ValueError(f'{name}: expected target {TARGETS[name]!r} not found in {p}; first columns: {header[:12]}')
display(pd.DataFrame([{'dataset': k, 'target': TARGETS[k], 'CSV': str(v)} for k, v in CSV_PATHS.items()]))

## Data preparation: the BRGM method, with dataset-specific context and horizon

All numeric columns other than `date` are included **in CSV order**. Training target is `OT` (ETT), `T (degC)` (Weather), and `319` (Electricity). The train-only scaler and 50/10/40 chronological split are identical to the previous pipeline. Validation/test receive `input_len + horizon` *context rows* from the previous split; no validation/test target values leak into model training or scaler fitting.

The legacy window class intentionally constructs `max(segment_len - input_len - horizon, 0)` windows (no `+1`); this notebook retains it for methodological fidelity. All 321 Electricity clients remain predictor inputs (they are **not** all presumed editable CF intervention features).

In [ ]:
# 4. Identical GRU layer names / state_dict structure as the saved piezometric models.
class SlidingWindowDataset(Dataset):
    def __init__(self, full_scaled, start, end, input_len, horizon, target_idx):
        self.full_scaled = full_scaled
        self.start, self.end = int(start), int(end)
        self.input_len, self.horizon, self.target_idx = input_len, horizon, target_idx
        self.num_samples = max((self.end-self.start)-input_len-horizon, 0)  # original, no +1
    def __len__(self): return self.num_samples
    def __getitem__(self, idx):
        t = self.start + idx
        x = self.full_scaled[t:t+self.input_len]
        y = self.full_scaled[t+self.input_len:t+self.input_len+self.horizon, self.target_idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32), t

class GRUForecaster(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, horizon):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True, dropout=0.0)
        self.fc = nn.Linear(hidden_size, horizon)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

@dataclass
class Prepared:
    name: str
    csv_path: Path
    df: pd.DataFrame
    feature_cols: list
    target_idx: int
    scaler: StandardScaler
    scaled: np.ndarray
    train_range: tuple
    val_range: tuple
    test_range: tuple
    datasets: dict
    data_signature: str
    config: GRUConfig
    def loader(self, split, shuffle=False):
        generator = None
        if shuffle:
            generator = torch.Generator().manual_seed(self.config.seed)
        return DataLoader(self.datasets[split], batch_size=self.config.batch_size,
                          shuffle=shuffle, generator=generator, num_workers=self.config.num_workers)
    def inverse_target(self, arr):
        # mathematically identical to inverse_transform of zeros with target column filled.
        return np.asarray(arr) * self.scaler.scale_[self.target_idx] + self.scaler.mean_[self.target_idx]

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(4*1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def prepare(name, path):
    cfg = DATASET_CFG[name]
    df = pd.read_csv(path)
    if 'date' not in df: raise ValueError(f'{name}: date column missing')
    df['date'] = pd.to_datetime(df['date'], errors='raise')
    if df['date'].isna().any(): raise ValueError(f'{name}: NaT dates')
    # mirror original: chronologically sort and reset index; NO automatic filling, dropping, interpolation
    df = df.sort_values('date').reset_index(drop=True)
    if df['date'].duplicated().any(): raise ValueError(f'{name}: duplicate dates in selected data')
    feat = [c for c in df.columns if c != 'date']
    missing = [c for c in feat if not pd.api.types.is_numeric_dtype(df[c])]
    if missing: raise TypeError(f'{name}: non-numeric predictors: {missing}')
    if TARGETS[name] not in feat: raise ValueError(f'{name}: target missing: {TARGETS[name]}')
    values = df[feat].values.astype(np.float32)
    if not np.isfinite(values).all():
        raise ValueError(f'{name}: non-finite numeric values; inspect CSV before training; no silent imputation')
    n = len(values); n_train = int(n*cfg.train_ratio); n_val = int(n*cfg.val_ratio)
    scaler = StandardScaler().fit(values[:n_train])
    scaled = scaler.transform(values).astype(np.float32)
    ranges = {'train':(0,n_train),
              'val':(n_train-cfg.input_len-cfg.horizon,n_train+n_val),
              'test':(n_train+n_val-cfg.input_len-cfg.horizon,n)}
    ds = {k:SlidingWindowDataset(scaled,*v,cfg.input_len,cfg.horizon,feat.index(TARGETS[name]))
          for k,v in ranges.items()}
    if min(map(len,ds.values())) == 0:
        raise ValueError(f'{name}: at least one empty split: {[(k,len(v)) for k,v in ds.items()]}')
    if EXPECTED_ROWS[name] != n or EXPECTED_FEATURES[name] != len(feat):
        raise ValueError(f'{name}: expected (rows, features)=({EXPECTED_ROWS[name]}, {EXPECTED_FEATURES[name]}), got {(n,len(feat))}. Verify version and update documented metadata only deliberately.')
    return Prepared(name, path, df, feat, feat.index(TARGETS[name]), scaler,
                    scaled, *[ranges[k] for k in ('train','val','test')], ds, sha256_file(path), cfg)

BUNDLES = {name:prepare(name,p) for name,p in CSV_PATHS.items()}
scan=[]
for name,b in BUNDLES.items():
    row={'dataset':name, 'rows':len(b.df),'features':len(b.feature_cols),
         'target': b.feature_cols[b.target_idx], 'target_idx':b.target_idx,
         'train_rows':int(len(b.df)*b.config.train_ratio),
         'val_rows':int(len(b.df)*b.config.val_ratio),
         'input_len':b.config.input_len,'horizon':b.config.horizon}
    row.update({f'{s}_windows':len(b.datasets[s]) for s in ('train','val','test')})
    scan.append(row)
display(pd.DataFrame(scan))

## Learning-rate grid search 

In [ ]:
# 5. Training/evaluation utilities matching old run_epoch, train_gru and pick_best_training_result.
def make_model(bundle):
    cfg = bundle.config
    return GRUForecaster(len(bundle.feature_cols),cfg.gru_hidden,cfg.gru_layers, cfg.horizon).to(cfg.device)

def run_epoch(model, loader, criterion, optimizer=None):
    model.train(optimizer is not None)
    total = 0.0; n = 0
    for xb,yb,_ in loader:
        device=next(model.parameters()).device
        xb=xb.to(device); yb=yb.to(device)
        with torch.set_grad_enabled(optimizer is not None):
            pred=model(xb); loss=criterion(pred,yb)
            if optimizer is not None:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()*xb.size(0); n += xb.size(0)
    return total / max(n,1)

@torch.no_grad()
def eval_original_scale(model, bundle, split, collect=False):
    model.eval()
    ys=[]; preds=[]; indices=[]
    for xb,yb,idx in bundle.loader(split):
        pr=model(xb.to(next(model.parameters()).device)).cpu().numpy()
        ys.append(bundle.inverse_target(yb.numpy()))
        preds.append(bundle.inverse_target(pr))
        indices.append(idx.numpy())
    yt=np.concatenate(ys); yp=np.concatenate(preds); ix=np.concatenate(indices)
    m={'mae':float(mean_absolute_error(yt.ravel(),yp.ravel())),
       'rmse':float(np.sqrt(mean_squared_error(yt.ravel(),yp.ravel()))),
       'r2':float(r2_score(yt.ravel(),yp.ravel())),
       'n_windows':int(len(ix))}
    if collect: return m,(yt,yp,ix)
    return m

def trial_meta(bundle,lr):
    cfg = bundle.config
    # Exclude hardware from identity: device may change at reload without changing numeric protocol.
    settings={k:v for k,v in asdict(cfg).items() if k!='device'}
    settings['gru_lrs']=list(cfg.gru_lrs)
    return dict(dataset=bundle.name,csv_sha256=bundle.data_signature,
                feature_cols=bundle.feature_cols,target=TARGETS[bundle.name],
                lr=float(lr),settings=settings,architecture='GRUForecaster/gru+fc')

def train_one_lr(bundle, lr):
    cfg = bundle.config
    set_seed(cfg.seed)  # identical initialization seed for each original grid-search trial
    model=make_model(bundle)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion=nn.L1Loss()
    tr_loader=bundle.loader('train',shuffle=True); va_loader=bundle.loader('val')
    best_state=None; best_val=float('inf'); wait=0; best_epoch=None
    hist={'train':[],'val':[]}
    for ep in range(1,cfg.gru_epochs+1):
        tr=run_epoch(model,tr_loader,criterion,optimizer)
        va=run_epoch(model,va_loader,criterion)
        hist['train'].append(float(tr));hist['val'].append(float(va))
        print(f'    lr={lr:g} epoch={ep:03d} train={tr:.6f} val={va:.6f}',flush=True)
        if va < best_val - cfg.min_delta:
            best_val=va
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            best_epoch=ep;wait=0
        else:
            wait+=1
            if wait >= cfg.patience:
                print('    Early stopping.',flush=True);break
    if best_state is None:raise RuntimeError('No valid checkpoint; check nonfinite data/loss')
    model.load_state_dict(best_state)
    val=eval_original_scale(model,bundle,'val')
    return model,hist,{'best_val_loss':float(best_val),'best_epoch':int(best_epoch),
                       'val_mae':val['mae'],'val_rmse':val['rmse'],
                       'val_r2':val['r2'],'n_val_windows':val['n_windows']}

def save_json(obj,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,default=str),encoding='utf-8')
    tmp.replace(path)

def trial_folder(name,lr):return OUTPUT_ROOT/'datasets'/name/'lr_trials'/lr_tag(lr)

def run_or_reload_trial(bundle,lr,overwrite=False):
    cfg = bundle.config
    folder=trial_folder(bundle.name,lr)
    ck=folder/'best_epoch_state.pth'; info=folder/'trial.json'
    wanted=trial_meta(bundle,lr)
    if ck.exists() and info.exists() and not overwrite:
        saved=json.loads(info.read_text())
        if saved['identity']!=wanted:
            raise ValueError(f'Incompatible saved trial at {folder}. Data/settings changed; use new OUTPUT_ROOT or set OVERWRITE_EXISTING=True.')
        model=make_model(bundle)
        model.load_state_dict(torch.load(ck,map_location=cfg.device,weights_only=True))
        print(f'  Reused completed LR {lr:g} for {bundle.name}')
        return model,saved['history'],saved['metrics']
    model,hist,metrics=train_one_lr(bundle,lr)
    folder.mkdir(parents=True,exist_ok=True)
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},ck)
    save_json({'identity':wanted,'history':hist,'metrics':metrics},info)
    return model,hist,metrics

In [ ]:
# 6. Grid search and saving — each model's own historical length and horizon.
ALL_GRID, ALL_FORECAST = [], []
for name,bundle in BUNDLES.items():
    cfg = bundle.config
    d = OUTPUT_ROOT / 'datasets' / name
    d.mkdir(parents=True,exist_ok=True)
    print(f'\n===== {name}: {len(bundle.feature_cols)} features, target={TARGETS[name]}, '
          f'input={cfg.input_len}, horizon={cfg.horizon} =====', flush=True)
    trials=[]  # metadata ONLY; no 6-model GPU cache
    for lr in cfg.gru_lrs:
        print(f'  Trying LR = {lr:g}', flush=True)
        model,hist,metrics=run_or_reload_trial(bundle,lr,overwrite=OVERWRITE_EXISTING)
        trials.append((lr, metrics))
        ALL_GRID.append({'dataset':name,'input_len':cfg.input_len,'horizon':cfg.horizon,
                         'gru_lr':float(lr),**metrics})
        pd.DataFrame(ALL_GRID).to_csv(OUTPUT_ROOT/'grid_search_results.csv', index=False)
        del model, hist
        if cfg.device == 'cuda': torch.cuda.empty_cache()
    lr,metrics=min(trials, key=lambda x:(x[1]['val_mae'],x[1]['val_rmse'],x[1]['best_val_loss']))
    best_trial=trial_folder(name,lr)
    model=make_model(bundle)
    model.load_state_dict(torch.load(best_trial/'best_epoch_state.pth',map_location=cfg.device,weights_only=True))
    hist=json.loads((best_trial/'trial.json').read_text())['history']
    print(f'  Selected lr={lr:g}, validation MAE={metrics["val_mae"]:.6g}, best epoch={metrics["best_epoch"]}')
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()}, d/'best_gru_forecaster.pth')
    meta={'dataset':name,'csv_path':str(bundle.csv_path),'csv_sha256':bundle.data_signature,
          'feature_cols':bundle.feature_cols,'target_col':TARGETS[name],
          'target_idx':bundle.target_idx,'best_lr':float(lr),
          'selection_criterion':'original-scale validation MAE, then val RMSE, then scaled val L1',
          'best_epoch':int(metrics['best_epoch']),'best_val_loss_scaled':metrics['best_val_loss'],
          'val_mae':metrics['val_mae'],'val_rmse':metrics['val_rmse'],
          'configuration':asdict(cfg),'train_range':bundle.train_range,
          'val_range':bundle.val_range,'test_range':bundle.test_range,
          'train_windows':len(bundle.datasets['train']),
          'val_windows':len(bundle.datasets['val']),
          'test_windows':len(bundle.datasets['test'])}
    save_json(meta,d/'best_gru_metadata.json')
    np.savez(d/'scaler_parameters.npz',mean=bundle.scaler.mean_,scale=bundle.scaler.scale_,
             var=bundle.scaler.var_,n_samples_seen=bundle.scaler.n_samples_seen_,
             feature_cols=np.asarray(bundle.feature_cols,dtype=str))
    pd.DataFrame({'epoch':range(1,len(hist['train'])+1),
                  'train_scaled_L1':hist['train'],'val_scaled_L1':hist['val']}).to_csv(
        d/'best_gru_training_history.csv',index=False)
    test=eval_original_scale(model,bundle,'test')
    row={'dataset':name,'target_col':TARGETS[name],'n_features':len(bundle.feature_cols),
         'input_len':cfg.input_len,'horizon':cfg.horizon,
         'best_gru_lr':float(lr),'n_test_windows_full':test['n_windows'],
         'test_mae_full':test['mae'],'test_rmse_full':test['rmse'],'test_r2_full':test['r2']}
    ALL_FORECAST.append(row)
    pd.DataFrame(ALL_FORECAST).to_csv(OUTPUT_ROOT/'forecast_metrics.csv',index=False)
    print(f'  FULL test {test["n_windows"]} windows: MAE {test["mae"]:.6g}, '
          f'RMSE {test["rmse"]:.6g}, R² {test["r2"]:.6g}',flush=True)
    del trials, model
    if cfg.device == 'cuda': torch.cuda.empty_cache()

grid_df=pd.DataFrame(ALL_GRID)
forecast_df=pd.DataFrame(ALL_FORECAST)
print('Saved best models and metrics under',OUTPUT_ROOT)
display(grid_df.sort_values(['dataset','val_mae']))
display(forecast_df)

## Inspect saved GRUs and their test predictions

Plots show: training/validation MAE per epoch; six-LR validation MAE; one **fixed-index** test-window target forecast versus ground truth, inverse-transformed to target physical units. The plotted horizon is 24 steps for hourly datasets and 144 steps for Weather.

In [ ]:
# 7. Grid-search curves, train/val loss curves and a sample forecast in original target units.
for name,bundle in BUNDLES.items():
    cfg=bundle.config
    d=OUTPUT_ROOT/'datasets'/name
    if not (d/'best_gru_metadata.json').exists(): continue
    meta=json.loads((d/'best_gru_metadata.json').read_text())
    h=pd.read_csv(d/'best_gru_training_history.csv')
    g=pd.read_csv(OUTPUT_ROOT/'grid_search_results.csv')
    g=g[g.dataset==name].sort_values('gru_lr')
    fig,ax=plt.subplots(figsize=(6,3))
    ax.plot(h.epoch,h.train_scaled_L1,label='Train (scaled MAE)')
    ax.plot(h.epoch,h.val_scaled_L1,label='Validation (scaled MAE)')
    ax.set(title=f'{name}: selected LR {meta["best_lr"]:g}',xlabel='Epoch',ylabel='MAE (scaled)')
    ax.legend();fig.tight_layout();fig.savefig(d/'best_gru_training_curve.png',dpi=150);plt.show();plt.close(fig)
    fig,ax=plt.subplots(figsize=(6,3))
    ax.plot(np.arange(len(g)),g.val_mae,'o-')
    ax.set_xticks(np.arange(len(g)),[f'{v:g}' for v in g.gru_lr],rotation=45)
    ax.set(title=f'{name}: LR search',xlabel='Learning rate',ylabel='Validation MAE (original units)')
    fig.tight_layout();fig.savefig(d/'gru_lr_grid_search.png',dpi=150);plt.show();plt.close(fig)
    model=make_model(bundle)
    model.load_state_dict(torch.load(d/'best_gru_forecaster.pth',map_location=cfg.device,weights_only=True))
    test,raw=eval_original_scale(model,bundle,'test',collect=True)
    ys,prs,starts=raw
    k=min(5,len(starts)-1)  # fixed example, no cherry-picking
    fig,ax=plt.subplots(figsize=(8,3))
    ax.plot(np.arange(1,cfg.horizon+1),ys[k],label='Observed')
    ax.plot(np.arange(1,cfg.horizon+1),prs[k],label='GRU predicted')
    ax.set(title=f'{name}: full-test window #{k}',xlabel='Forecast step',ylabel=TARGETS[name])
    ax.legend();fig.tight_layout();fig.savefig(d/'test_forecast_example.png',dpi=150);plt.show();plt.close(fig)
    del model

## Independent model reload and reproducibility check

This cell can be run without training (skip grid-search cell) in a new session. It verifies each saved CSV hash, target/feature order, scaler parameters, split boundaries, architecture and **dataset-specific input length/horizon** before computing full-test predictions. `best_gru_forecaster.pth` contains the same `gru.*` and `fc.*` weight naming as the BRGM models.

In [ ]:
# 8. Can be run independently of step 6 (when checkpoints already exist).
RELOADED=[]
for name,bundle in BUNDLES.items():
    cfg=bundle.config
    d=OUTPUT_ROOT/'datasets'/name
    meta_path=d/'best_gru_metadata.json'
    ck=d/'best_gru_forecaster.pth'
    if not meta_path.exists() or not ck.exists():
        print(f'No trained model for {name}; skip.');continue
    m=json.loads(meta_path.read_text())
    assert m['csv_sha256']==bundle.data_signature, f'{name}: CSV bytes changed'
    assert m['feature_cols']==bundle.feature_cols, f'{name}: predictor order changed'
    assert m['target_col']==TARGETS[name] and m['target_idx']==bundle.target_idx
    assert list(m['train_range'])==list(bundle.train_range)
    assert list(m['val_range'])==list(bundle.val_range)
    assert list(m['test_range'])==list(bundle.test_range)
    for key in ('input_len','horizon','gru_hidden','gru_layers','train_ratio','val_ratio'):
        assert m['configuration'][key]==getattr(cfg,key), f'{name}: config differs: {key}'
    saved_scaler=np.load(d/'scaler_parameters.npz')
    assert np.array_equal(saved_scaler['feature_cols'],np.array(bundle.feature_cols,dtype=str))
    assert np.allclose(saved_scaler['mean'],bundle.scaler.mean_)
    assert np.allclose(saved_scaler['scale'],bundle.scaler.scale_)
    model=make_model(bundle)
    model.load_state_dict(torch.load(ck,map_location=cfg.device,weights_only=True))
    model.eval()
    val=eval_original_scale(model,bundle,'val')
    test=eval_original_scale(model,bundle,'test')
    RELOADED.append({'dataset':name,'best_gru_lr':m['best_lr'],
                     'test_windows':test['n_windows'],'val_mae':val['mae'],
                     'test_mae':test['mae'],'test_rmse':test['rmse'],'test_r2':test['r2'],
                     'model_path':str(ck)})
    del model
reloaded_df=pd.DataFrame(RELOADED)
display(reloaded_df)
checkdir=OUTPUT_ROOT/'best_model_re_evaluation';checkdir.mkdir(exist_ok=True)
reloaded_df.to_csv(checkdir/'best_models_test_re_evaluation.csv',index=False)
refpath=OUTPUT_ROOT/'forecast_metrics.csv'
if refpath.exists() and len(reloaded_df):
    ref=pd.read_csv(refpath)
    c=reloaded_df.merge(ref,on='dataset',how='left',suffixes=('','_saved'))
    for key,prior in [('test_mae','test_mae_full'),('test_rmse','test_rmse_full'),('test_r2','test_r2_full')]:
        c['delta_'+key]=c[key]-c[prior]
        assert np.allclose(c[key],c[prior],rtol=1e-4,atol=1e-5),f'Metrics inconsistent: {key}'
    display(c[['dataset','delta_test_mae','delta_test_rmse','delta_test_r2']])
    c.to_csv(checkdir/'best_models_test_re_evaluation_comparison.csv',index=False)
print('Reload checks complete:',checkdir)